# CardioVision: data inspection

Inspect the three source datasets, including source-level missing-value markers, schema, descriptive statistics, and target balance. The final cell writes `result/01_inspection_summary.md`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data" / "cardio_train.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/cardio_train.csv from the notebook working directory.")
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

cleveland_columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target",
]
cardio = pd.read_csv(DATA_DIR / "cardio_train.csv", sep=";")
framingham = pd.read_csv(DATA_DIR / "framingham.csv")
cleveland_source = (DATA_DIR / "UCI" / "processed.cleveland.data").read_text(encoding="utf-8")
cleveland_question_marks = cleveland_source.count("?")
cleveland = pd.read_csv(
    DATA_DIR / "UCI" / "processed.cleveland.data",
    header=None,
    names=cleveland_columns,
    na_values=["?"],
)

datasets = {
    "Cardio Train": (cardio, "cardio"),
    "Framingham": (framingham, "TenYearCHD"),
    "Cleveland": (cleveland, "target"),
}
print(f"Project root: {PROJECT_ROOT}")
print(f"Literal '?' placeholders in Cleveland source: {cleveland_question_marks}")

Project root: C:\Users\kunug\OneDrive\Desktop\cardioVision
Literal '?' placeholders in Cleveland source: 6


## Shapes, types, samples, missingness, and descriptive statistics

In [2]:
for name, (df, target) in datasets.items():
    print(f"\n{'=' * 24} {name} {'=' * 24}")
    print("Shape:", df.shape)
    print("Column data types:")
    display(df.dtypes.to_frame("dtype"))
    print("Sample rows:")
    display(df.head())
    print("Missing-value counts:")
    display(df.isnull().sum().to_frame("missing_count"))
    print("Descriptive statistics:")
    display(df.describe(include="all").T)


======================== Cardio Train ========================
Shape: (70000, 13)
Column data types:


,dtype
id,int64
age,int64
gender,int64
height,int64
weight,float64
ap_hi,int64
ap_lo,int64
cholesterol,int64
gluc,int64
smoke,int64


Sample rows:


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


Missing-value counts:


...

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


Missing-value counts:


,missing_count
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## Target class distributions and balance

The Cleveland target is retained as its original five-level outcome (0 = no disease; 1–4 = disease severity categories).

In [3]:
target_summaries = {}
for name, (df, target) in datasets.items():
    counts = df[target].value_counts(dropna=False).sort_index()
    percentages = (counts / len(df) * 100).round(2)
    target_summaries[name] = pd.DataFrame({"count": counts, "percent": percentages})
    print(f"\n{name} — target: {target} — class counts and percentages")
    display(target_summaries[name])
    nonzero = counts[counts > 0]
    print(f"Largest-to-smallest class ratio: {nonzero.max() / nonzero.min():.2f}")


Cardio Train — target: cardio — class counts and percentages


,count,percent
cardio,,
0,35021,50.03
1,34979,49.97


Largest-to-smallest class ratio: 1.00

Framingham — target: TenYearCHD — class counts and percentages


,count,percent
TenYearCHD,,
0,3596,84.81
1,644,15.19


Largest-to-smallest class ratio: 5.58

Cleveland — target: target — class counts and percentages


,count,percent
target,,
0,164,54.13
1,55,18.15
2,36,11.88
3,35,11.55
4,13,4.29


Largest-to-smallest class ratio: 12.62


## Write inspection summary

In [4]:
summary = [
    "# Data inspection summary",
    "",
    "Generated from the source files by `notebook/01_data_inspection.ipynb`.",
    "",
    f"- The processed Cleveland source contains **{cleveland_question_marks}** literal `?` missing-value flags; these were parsed as nulls for inspection.",
]
for name, (df, target) in datasets.items():
    missing = df.isna().sum()
    missing_columns = missing[missing > 0]
    dtypes = ", ".join(f"`{col}`: `{dtype}`" for col, dtype in df.dtypes.items())
    target_balance = "\n".join([
        "| Class | Count | Percent |",
        "|---|---:|---:|",
        *[
            f"| {label} | {int(row['count'])} | {row['percent']:.2f}% |"
            for label, row in target_summaries[name].iterrows()
        ],
    ])
    summary.extend([
        "",
        f"## {name}",
        "",
        f"- **Shape:** {df.shape[0]:,} rows × {df.shape[1]} columns.",
        f"- **Target:** `{target}`; class distribution (count and percent):",
        target_balance,
        "- **Missing values:** " + (
            "; ".join(f"`{col}`: {int(count)}" for col, count in missing_columns.items())
            if len(missing_columns) else "none detected"
        ) + f" (total {int(missing.sum())}).",
        f"- **Column dtypes:** {dtypes}.",
    ])
summary.extend([
    "",
    "## Cross-dataset data-type discrepancies",
    "",
    "- The datasets use different field names and units for related attributes: cardio `age` is recorded in days, while Framingham and Cleveland `age` are in years; blood pressure and cholesterol fields also have dataset-specific names.",
    "- Cardio Train is predominantly integer-valued with floating-point `weight`; Framingham has a mix of integer and floating-point columns.",
    "- Cleveland's headerless file is parsed mostly as `float64` because its numeric fields contain decimal-formatted values and `?` placeholders; `target` is integer-valued. Encoded clinical categories are numeric codes, not continuous measurements.",
    "- Missingness is explicitly represented by `?` in Cleveland (`ca` and `thal`) and by blank values in Framingham. Cardio Train has no nulls in its source columns.",
    "",
])
summary_path = RESULT_DIR / "01_inspection_summary.md"
summary_path.write_text("\n".join(summary), encoding="utf-8")
print(f"Wrote {summary_path}")
print("\n".join(summary))

Wrote C:\Users\kunug\OneDrive\Desktop\cardioVision\result\01_inspection_summary.md
# Data inspection summary

Generated from the source files by `notebook/01_data_inspection.ipynb`.

- The processed Cleveland source contains **6** literal `?` missing-value flags; these were parsed as nulls for inspection.

## Cardio Train

- **Shape:** 70,000 rows × 13 columns.
- **Target:** `cardio`; class distribution (count and percent):
| Class | Count | Percent |
|---|---:|---:|
| 0 | 35021 | 50.03% |
| 1 | 34979 | 49.97% |
- **Missing values:** none detected (total 0).
- **Column dtypes:** `id`: `int64`, `age`: `int64`, `gender`: `int64`, `height`: `int64`, `weight`: `float64`, `ap_hi`: `int64`, `ap_lo`: `int64`, `cholesterol`: `int64`, `gluc`: `int64`, `smoke`: `int64`, `alco`: `int64`, `active`: `int64`, `cardio`: `int64`.

## Framingham

- **Shape:** 4,240 rows × 16 columns.
- **Target:** `TenYearCHD`; class distribution (count and percent):
| Class | Count | Percent |
|---|---:|---:|
| 0 | 35